In [0]:
!pip install pydexcom

In [0]:
%restart_python

In [0]:
    import json
    with open("/Workspace/Users/bartosz.moszczynski@gmail.com/Project1/DatabriksProjects/secret.json") as f:
        secrets = json.load(f)

## get data from dexcom site

In [0]:
from pydexcom import Dexcom
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, StringType

dexcom = Dexcom( **secrets) 
readings = dexcom.get_glucose_readings(minutes=21*60)  

data = [
    {
        "timestamp": r.datetime,      # from API
        "mg_dl": r.value,
        "trend": r.trend,
        # "trend_rate_mgdl_min": r.trend_rate,  # sometimes None
        "raw": getattr(r, "raw_value", None)
    }
    for r in readings
]

# Convert directly to Spark DataFrame

# Define the schema
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("mg_dl", IntegerType(), True),
    StructField("trend", StringType(), True),
    StructField("raw", IntegerType(), True)
])

df_spark = spark.createDataFrame(data,schema)

# (Optional) register as a temp view so you can query with SQL
df_spark.createOrReplaceTempView("dexcom_readings")

df_spark.write.mode("overwrite").csv("dbfs:/Volumes/workspace/training_samples/test/dexcom_readings/", header=True)


## MERGE INTO dexcom_readings_delta AS target USING dexcom_readings 

In [0]:
%sql
MERGE INTO dexcom_readings_delta AS target
USING dexcom_readings AS source
ON target.timestamp = source.timestamp
WHEN MATCHED THEN UPDATE SET
  target.mg_dl   = source.mg_dl,
  target.trend   = source.trend,
  target.raw     = source.raw,
  target.load_ts = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
  timestamp, mg_dl, trend, raw, load_ts
) VALUES (
  source.timestamp, source.mg_dl, source.trend, source.raw, current_timestamp()
);
